# Import Fine-tuned LLaMA 3 models on SageMaker JumpStart to private model hub

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook.

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

---
In this demo notebook, we demonstrate how to use the SageMaker Python SDK to deploy pre-trained Llama 3 model as well as fine-tune it for your dataset in domain adaptation or instruction tuning format. We will then import the model into a jumpstart private model hub.

---

### Model License information
---
To perform inference on these models, you need to pass custom_attributes='accept_eula=true' as part of header. This means you have read and accept the end-user-license-agreement (EULA) of the model. EULA can be found in model card description or from https://ai.meta.com/resources/models-and-libraries/llama-downloads/. By default, this notebook sets custom_attributes='accept_eula=false', so all inference requests will fail until you explicitly change this custom attribute.

Note: Custom_attributes used to pass EULA are key/value pairs. The key and value are separated by '=' and pairs are separated by ';'. If the user passes the same key more than once, the last value is kept and passed to the script handler (i.e., in this case, used for conditional logic). For example, if 'accept_eula=false; accept_eula=true' is passed to the server, then 'accept_eula=true' is kept and passed to the script handler.

---

### Set up

---
We begin by installing and upgrading necessary packages. Restart the kernel after executing the cell below for the first time.

---

In [19]:
!pip install --upgrade sagemaker datasets

In [20]:
import warnings
import logging


# Suppress warnings
warnings.filterwarnings("ignore")

# Suppress INFO messages
logging.getLogger().setLevel(logging.ERROR)

## Deploy Pre-trained Model

---

First we will deploy the Llama-2 model as a SageMaker endpoint. To train/deploy 8B and 70B models, please change model_id to "meta-textgeneration-llama-3-8b" and "meta-textgeneration-llama-3-70b" respectively.

---

In [21]:
model_id, model_version = "meta-textgeneration-llama-3-8b", "2.*"

In [22]:
from sagemaker.jumpstart.model import JumpStartModel

pretrained_model = JumpStartModel(model_id=model_id, model_version=model_version)
# Please change the following line to have accept_eula = True
pretrained_predictor = pretrained_model.deploy(accept_eula=True)

-----------!

## Invoke the endpoint

---
Next, we invoke the endpoint with some sample queries. Later, in this notebook, we will fine-tune this model with a custom dataset and carry out inference using the fine-tuned model. We will also show comparison between results obtained via the pre-trained and the fine-tuned models.

---

In [23]:
def print_response(payload, response):
    print(payload["inputs"])
    print(f"> {response.get('generated_text')}")
    print("\n==================================\n")

In [24]:
payload = {
    "inputs": "I believe the meaning of life is",
    "parameters": {
        "max_new_tokens": 64,
        "top_p": 0.9,
        "temperature": 0.6,
        "return_full_text": False,
    },
}
try:
    response = pretrained_predictor.predict(
        payload, custom_attributes="accept_eula=false"
    )
    print_response(payload, response)
except Exception as e:
    print(e)

I believe the meaning of life is
>  to be happy. I believe that happiness is the most important thing in life. I believe that happiness is the most important thing in life. I believe that happiness is the most important thing in life. I believe that happiness is the most important thing in life. I believe that happiness is the most important thing in life.




---
To learn about additional use cases of pre-trained model, please checkout the notebook [Text completion: Run Llama 3 models in SageMaker JumpStart](https://github.com/aws/amazon-sagemaker-examples/blob/main/introduction_to_amazon_algorithms/jumpstart-foundation-models/llama-3-text-completion.ipynb).

---

## Dataset preparation for fine-tuning

---

You can fine-tune on the dataset with domain adaptation format or instruction tuning format. Please find more details in the section [Dataset instruction](#Dataset-instruction). In this demo, we will use a subset of [Dolly dataset](https://huggingface.co/datasets/databricks/databricks-dolly-15k) in an instruction tuning format. Dolly dataset contains roughly 15,000 instruction following records for various categories such as question answering, summarization, information extraction etc. It is available under Apache 2.0 license. We will select the summarization examples for fine-tuning.


Training data is formatted in JSON lines (.jsonl) format, where each line is a dictionary representing a single data sample. All training data must be in a single folder, however it can be saved in multiple jsonl files. The training folder can also contain a template.json file describing the input and output formats.

To train your model on a collection of unstructured dataset (text files), please see the section [Example fine-tuning with Domain-Adaptation dataset format](#Example-fine-tuning-with-Domain-Adaptation-dataset-format) in the Appendix.

---

In [25]:
from datasets import load_dataset

dolly_dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

# To train for question answering/information extraction, you can replace the assertion in next line to example["category"] == "closed_qa"/"information_extraction".
summarization_dataset = dolly_dataset.filter(
    lambda example: example["category"] == "summarization"
)
summarization_dataset = summarization_dataset.remove_columns("category")

# We split the dataset into two where test data is used to evaluate at the end.
train_and_test_dataset = summarization_dataset.train_test_split(test_size=0.1)

# Dumping the training data to a local file to be used for training.
train_and_test_dataset["train"].to_json("train.jsonl")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

2127555

In [26]:
train_and_test_dataset["train"][0]

{'instruction': 'Using examples from the text, list some popular songs from the album For Bitter or Worse.',
 'context': 'For Bitter or Worse is the sixth studio album from the Dutch singer Anouk. The album was released on 18 September 2009, via the record label EMI.\n\nThe first single from the album, "Three Days in a Row" was released in August. It reached the top of the Netherlands charts in September 2009, making it Anouk\'s first number one in the country. In June of the same year, one of the songs recorded for the album, "Today", was released as promo material. It was so successful that, despite never being released as an official single, the song reached number 50 in the Dutch chart. The second single Woman, was sent to radio stations at the end of October 2009. After just one day the single was at number one on airplay chart. The single was released physically on 24 November 2009.',
 'response': 'Popular songs from the album For Bitter or Worse include "Three Days in a Row" and

---
Next, we create a prompt template for using the data in an instruction / input format for the training job (since we are instruction fine-tuning the model in this example), and also for inferencing the deployed endpoint.

---

In [27]:
import json

template = {
    "prompt": "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{context}\n\n",
    "completion": " {response}",
}
with open("template.json", "w") as f:
    json.dump(template, f)

### Upload dataset to S3
---

We will upload the prepared dataset to S3 which will be used for fine-tuning.

---

In [28]:
from sagemaker.s3 import S3Uploader
import sagemaker
import random

output_bucket = sagemaker.Session().default_bucket()
default_bucket_prefix = sagemaker.Session().default_bucket_prefix

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    train_data_location = f"s3://{output_bucket}/{default_bucket_prefix}/dolly_dataset"
else:
    train_data_location = f"s3://{output_bucket}/dolly_dataset"

local_data_file = "train.jsonl"
S3Uploader.upload(local_data_file, train_data_location)
S3Uploader.upload("template.json", train_data_location)
print(f"Training data: {train_data_location}")

Training data: s3://sagemaker-us-east-1-891376962744/dolly_dataset


## Train the model
---
Next, we fine-tune the LLaMA 3 8B model on the summarization dataset from Dolly. Finetuning scripts are based on scripts provided by [this repo](https://github.com/facebookresearch/llama-recipes/tree/main). To learn more about the fine-tuning scripts, please checkout section [5. Few notes about the fine-tuning method](#5.-Few-notes-about-the-fine-tuning-method). For a list of supported hyper-parameters and their default values, please see section [3. Supported Hyper-parameters for fine-tuning](#3.-Supported-Hyper-parameters-for-fine-tuning).

---

In [29]:
from sagemaker.jumpstart.estimator import JumpStartEstimator


estimator = JumpStartEstimator(
    model_id=model_id,
    model_version=model_version,
    environment={"accept_eula": "true"},  # Please change {"accept_eula": "true"}
    disable_output_compression=True,
    instance_type="ml.g5.12xlarge",  # For Llama-3-70b, add instance_type = "ml.g5.48xlarge"
)
# By default, instruction tuning is set to false. Thus, to use instruction tuning dataset you use
estimator.set_hyperparameters(
    instruction_tuned="True", epoch="1", max_input_length="1024"
)
estimator.fit({"training": train_data_location})

# Get the model artifacts path after training
model_artifacts = estimator.model_data

2025-03-10 17:25:55 Starting - Starting the training job
2025-03-10 17:25:55 Pending - Training job waiting for capacity............
2025-03-10 17:27:43 Pending - Preparing the instances for training...
2025-03-10 17:28:17 Downloading - Downloading input data...........................
2025-03-10 17:32:50 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2025-03-10 17:32:53,988 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-10 17:32:54,026 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-10 17:32:54,034 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-10 17:32:54,036 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-10 17:33:02,718 sagemaker-training-toolkit INFO     Installing d

In [30]:
model_path = model_artifacts["S3DataSource"]["S3Uri"]
model_path

's3://sagemaker-us-east-1-891376962744/meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/'

Studio Kernel Dying issue:  If your studio kernel dies and you lose reference to the estimator object, please see section [6. Studio Kernel Dead/Creating JumpStart Model from the training Job](#6.-Studio-Kernel-Dead/Creating-JumpStart-Model-from-the-training-Job) on how to deploy endpoint using the training job name and the model id. 


### Deploy the fine-tuned model
---
Next, we deploy fine-tuned model. We will compare the performance of fine-tuned and pre-trained model.

---

In [31]:
finetuned_predictor = estimator.deploy()

----------!

### Evaluate the pre-trained and fine-tuned model
---
Next, we use the test data to evaluate the performance of the fine-tuned model and compare it with the pre-trained model. 

---

In [32]:
import pandas as pd
from IPython.display import display, HTML

test_dataset = train_and_test_dataset["test"]

(
    inputs,
    ground_truth_responses,
    responses_before_finetuning,
    responses_after_finetuning,
) = (
    [],
    [],
    [],
    [],
)


def predict_and_print(datapoint):
    # For instruction fine-tuning, we insert a special key between input and output
    input_output_demarkation_key = "\n\n### Response:\n"

    payload = {
        "inputs": template["prompt"].format(
            instruction=datapoint["instruction"], context=datapoint["context"]
        )
        + input_output_demarkation_key,
        "parameters": {"max_new_tokens": 100},
    }
    inputs.append(payload["inputs"])
    ground_truth_responses.append(datapoint["response"])
    # Please change the following line to "accept_eula=true"
    pretrained_response = pretrained_predictor.predict(
        payload, custom_attributes="accept_eula=false"
    )
    responses_before_finetuning.append(pretrained_response.get("generated_text"))
    # Fine Tuned Llama 3 models doesn't required to set "accept_eula=true"
    finetuned_response = finetuned_predictor.predict(payload)
    responses_after_finetuning.append(finetuned_response.get("generated_text"))


try:
    for i, datapoint in enumerate(test_dataset.select(range(5))):
        predict_and_print(datapoint)

    df = pd.DataFrame(
        {
            "Inputs": inputs,
            "Ground Truth": ground_truth_responses,
            "Response from non-finetuned model": responses_before_finetuning,
            "Response from fine-tuned model": responses_after_finetuning,
        }
    )
    display(HTML(df.to_html()))
except Exception as e:
    print(e)

,Inputs,Ground Truth,Response from non-finetuned model,Response from fine-tuned model
0,"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGive me a summary of Dataphor based on this text\n\n### Input:\nDataphor is an open-source truly-relational database management system (RDBMS) and its accompanying user interface technologies, which together are designed to provide highly declarative software application development. The Dataphor Server has its own storage engine or it can be a virtual, or federated, DBMS, meaning that it can utilize other database engines for storage.\n\nDataphor has been praised for its adherence to relational principles, more closely so than any SQL product.\n\n\n\n### Response:\n",Dataphor is an open-source database management system that provides a declarative software application development that has been praised for its adherence to relational principles. The Dataphor Server has its own storage engine but it can also utilize other database engines for storage that are virtual or federated DBMS.,"Dataphor is an open-source truly-relational database management system (RDBMS) and its accompanying user interface technologies, which together are designed to provide highly declarative software application development. The Dataphor Server has its own storage engine or it can be a virtual, or federated, DBMS, meaning that it can utilize other database engines for storage. Dataphor has been praised for its adherence to relational principles, more closely so than any SQL product.","Dataphor is an open-source truly-relational database management system (RDBMS) and its accompanying user interface technologies, which together are designed to provide highly declarative software application development. The Dataphor Server has its own storage engine or it can be a virtual, or federated, DBMS, meaning that it can utilize other database engines for storage. Dataphor has been praised for its adherence to relational principles, more closely so than any SQL product."
1,"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nHow do you make wine?\n\n### Input:\nWinemaking (also wine making) or vinification is the production of wine, starting with the selection of the fruit, its fermentation into alcohol, and the bottling of the finished liquid. The history of wine-making stretches over millennia. The science of wine and winemaking is known as oenology. A winemaker may also be called a vintner. The growing of grapes is viticulture and there are many varieties of grapes.\n\nWinemaking can be divided into two general categories: still wine production (without carbonation) and sparkling wine production (with carbonation – natural or injected). Red wine, white wine, and rosé are the other main categories. Although most wine is made from grapes, it may also be made from other plants. (See fruit wine.) Other similar light alcoholic drinks (as opposed to beer or spirits) include mead, made by fermenting honey and water, cider (""apple cider""), made by fermenting the juice of apples, and perry (""pear cider""), made by fermenting the juice of pears, and kumis, made of fermented mare's milk.\n\n\n\n### Response:\n","Winemaking is the process of using fruit for fermentation into alcohol. The science of winemaking is called enology. There are two general types of wine production: still wine and sparkling wine. The three categories of wine are red, white, and rose. Most wine is made from grapes but others can include apples, pears, and honey.","I make wine by fermenting grapes. I use a process called maceration, which involves crushing the grapes and allowing them to sit in a container for a period of time. This allows the grapes to release their juices and the yeast to start the ferm

### Clean up resources

In [33]:
# Delete resources
pretrained_predictor.delete_model()
pretrained_predictor.delete_endpoint()
finetuned_predictor.delete_model()
finetuned_predictor.delete_endpoint()

### Import the fine tuned model to Jumpstart private model hub
---
Next, we will prepare the model artifact and upload it into the Sagemaker's default S3 bucket 

---

In [34]:
import utils

# download model artifacts
utils.download_model_from_s3(model_path)

# Create the model.tar.gz
# this process take a while
utils.package_llama_model("downloaded_model", "model.tar.gz")

Downloading: meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/__script_info__.json to downloaded_model/__script_info__.json
Downloading: meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/config.json to downloaded_model/config.json
Downloading: meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/generation_config.json to downloaded_model/generation_config.json
Downloading: meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/inference.py to downloaded_model/inference.py
Downloading: meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/model-00001-of-00004.safetensors to downloaded_model/model-00001-of-00004.safetensors
Downloading: meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/model-00002-of-00004.safetensors to downloaded_model/model-00002-of-00004.safetensors
Downloading: meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output/model/model-00003-of-00004.safetensors to downloa

In [36]:
# upload model artifact to S3
# we will upload it to the output folder in model_path variable
# i.e. s3://sagemaker-us-east-1-891376962744/meta-textgeneration-llama-3-8b-2025-03-10-14-09-25-370/output/model/ for my environment


s3_artifact_uri = utils.upload_model_to_s3(
    bucket="sagemaker-us-east-1-891376962744",
    prefix="meta-textgeneration-llama-3-8b-2025-03-10-17-25-53-293/output",
)

s3_artifact_uri

Model successfully uploaded to: s3://sagemaker-us-east-1-891376962744/meta-textgeneration-llama-3-8b-2025-03-10-14-09-25-370/output/model.tar.gz


's3://sagemaker-us-east-1-891376962744/meta-textgeneration-llama-3-8b-2025-03-10-14-09-25-370/output/model.tar.gz'

In [37]:
s3_artifact_uri

In [39]:
# define the private mode hub details to create one.
import boto3
import sagemaker
from sagemaker import Session
from sagemaker.jumpstart.hub.hub import Hub

sess = sagemaker.Session()

default_bucket = sess.default_bucket()
print(default_bucket)

HUB_NAME = "Custom-Model-HubZ"
HUB_DISPLAY_NAME = "Custom-Model-HubZ"

REGION = "us-east-1"


sm_client = boto3.client("sagemaker")
session = Session(sagemaker_client=sm_client)
session.get_caller_identity_arn()

'arn:aws:iam::891376962744:role/service-role/SageMaker-ExecutionRole-20240626T145306'

In [40]:
hub = Hub(hub_name=HUB_NAME, sagemaker_session=session, bucket_name=default_bucket)

try:
    hub.create(
        description="This is a Curated Hub. Replace this with a description which explains the purpose of this Hub.",
        display_name=HUB_DISPLAY_NAME,
    )
    print(f"Successfully created Hub with name {HUB_NAME} in {REGION}")
except Exception as e:
    if "ResourceInUse" in str(e):
        print(f"A hub with the name {HUB_NAME} already exists in your account.")
    else:
        raise e

A hub with the name Custom-Model-HubZ already exists in your account.


In [41]:
## Retrieve to update the HubContentDocument with the fine-tuned model artifacts
## you can use the below function

# hub_content = utils.get_hub_content_document()

In [42]:
## updating the object to point to our custom model artifact

hub_model_dict = {
    "Url": "https://ai.meta.com/resources/models-and-libraries/llama-downloads/",
    "MinSdkVersion": "2.225.0",
    "TrainingSupported": True,
    "IncrementalTrainingSupported": True,
    "HostingEcrSpecs": {
        "Framework": "djl-lmi",
        "FrameworkVersion": "0.28.0",
        "PyVersion": "py310",
    },
    "HostingArtifactUri": s3_artifact_uri,
    "HostingScriptUri": s3_artifact_uri,
    "HostingUseScriptUri": False,
    "HostingEulaUri": "s3://jumpstart-cache-prod-us-east-1/fmhMetadata/eula/llama3Eula.txt",
    "InferenceDependencies": [],
    "TrainingDependencies": [
        "accelerate==0.33.0",
        "bitsandbytes==0.39.1",
        "black==23.7.0",
        "brotli==1.0.9",
        "datasets==2.14.1",
        "docstring-parser==0.16",
        "fire==0.5.0",
        "huggingface-hub==0.24.2",
        "inflate64==0.3.1",
        "loralib==0.1.1",
        "multivolumefile==0.2.3",
        "mypy-extensions==1.0.0",
        "nvidia-cublas-cu12==12.1.3.1",
        "nvidia-cuda-cupti-cu12==12.1.105",
        "nvidia-cuda-nvrtc-cu12==12.1.105",
        "nvidia-cuda-runtime-cu12==12.1.105",
        "nvidia-cudnn-cu12==8.9.2.26",
        "nvidia-cufft-cu12==11.0.2.54",
        "nvidia-curand-cu12==10.3.2.106",
        "nvidia-cusolver-cu12==11.4.5.107",
        "nvidia-cusparse-cu12==12.1.0.106",
        "nvidia-nccl-cu12==2.19.3",
        "nvidia-nvjitlink-cu12==12.3.101",
        "nvidia-nvtx-cu12==12.1.105",
        "pathspec==0.11.1",
        "peft==0.4.0",
        "py7zr==0.20.5",
        "pybcj==1.0.1",
        "pycryptodomex==3.18.0",
        "pyppmd==1.0.0",
        "pyzstd==0.15.9",
        "safetensors==0.4.2",
        "sagemaker_jumpstart_huggingface_script_utilities==1.2.7",
        "sagemaker_jumpstart_script_utilities==1.1.9",
        "scipy==1.11.1",
        "shtab==1.7.1",
        "termcolor==2.3.0",
        "texttable==1.6.7",
        "tokenize-rt==5.1.0",
        "tokenizers==0.19.1",
        "torch==2.2.0",
        "transformers==4.43.1",
        "triton==2.2.0",
        "trl==0.8.1",
        "typing-extensions==4.8.0",
        "tyro==0.7.3",
    ],
    "Hyperparameters": [
        {
            "Name": "int8_quantization",
            "Type": "text",
            "Default": "False",
            "Options": ["True", "False"],
            "Scope": "algorithm",
        },
        {
            "Name": "enable_fsdp",
            "Type": "text",
            "Default": "True",
            "Options": ["True", "False"],
            "Scope": "algorithm",
        },
        {
            "Name": "epoch",
            "Type": "int",
            "Default": 5,
            "Min": 1,
            "Max": 1000,
            "Scope": "algorithm",
        },
        {
            "Name": "learning_rate",
            "Type": "float",
            "Default": 0.0001,
            "Min": 1e-08,
            "Max": 1,
            "Scope": "algorithm",
        },
        {"Name": "lora_r", "Type": "int", "Default": 8, "Min": 1, "Scope": "algorithm"},
        {
            "Name": "lora_alpha",
            "Type": "int",
            "Default": 32,
            "Min": 1,
            "Scope": "algorithm",
        },
        {
            "Name": "target_modules",
            "Type": "text",
            "Default": "q_proj,v_proj",
            "Scope": "algorithm",
        },
        {
            "Name": "lora_dropout",
            "Type": "float",
            "Default": 0.05,
            "Min": 0,
            "Max": 1,
            "Scope": "algorithm",
        },
        {
            "Name": "instruction_tuned",
            "Type": "text",
            "Default": "False",
            "Options": ["True", "False"],
            "Scope": "algorithm",
        },
        {
            "Name": "chat_dataset",
            "Type": "text",
            "Default": "False",
            "Options": ["True", "False"],
            "Scope": "algorithm",
        },
        {
            "Name": "add_input_output_demarcation_key",
            "Type": "text",
            "Default": "True",
            "Options": ["True", "False"],
            "Scope": "algorithm",
        },
        {
            "Name": "per_device_train_batch_size",
            "Type": "int",
            "Default": 4,
            "Min": 1,
            "Max": 1000,
            "Scope": "algorithm",
        },
        {
            "Name": "per_device_eval_batch_size",
            "Type": "int",
            "Default": 1,
            "Min": 1,
            "Max": 1000,
            "Scope": "algorithm",
        },
        {
            "Name": "max_train_samples",
            "Type": "int",
            "Default": -1,
            "Min": -1,
            "Scope": "algorithm",
        },
        {
            "Name": "max_val_samples",
            "Type": "int",
            "Default": -1,
            "Min": -1,
            "Scope": "algorithm",
        },
        {
            "Name": "seed",
            "Type": "int",
            "Default": 10,
            "Min": 1,
            "Max": 1000,
            "Scope": "algorithm",
        },
        {
            "Name": "max_input_length",
            "Type": "int",
            "Default": -1,
            "Min": -1,
            "Scope": "algorithm",
        },
        {
            "Name": "validation_split_ratio",
            "Type": "float",
            "Default": 0.2,
            "Min": 0,
            "Max": 1,
            "Scope": "algorithm",
        },
        {
            "Name": "train_data_split_seed",
            "Type": "int",
            "Default": 0,
            "Min": 0,
            "Scope": "algorithm",
        },
        {
            "Name": "preprocessing_num_workers",
            "Type": "text",
            "Default": "None",
            "Scope": "algorithm",
        },
        {
            "Name": "sagemaker_submit_directory",
            "Type": "text",
            "Default": "/opt/ml/input/data/code/sourcedir.tar.gz",
            "Scope": "container",
        },
        {
            "Name": "sagemaker_program",
            "Type": "text",
            "Default": "transfer_learning.py",
            "Scope": "container",
        },
        {
            "Name": "sagemaker_container_log_level",
            "Type": "text",
            "Default": "20",
            "Scope": "container",
        },
    ],
    "TrainingScriptUri": "s3://jumpstart-cache-prod-us-east-1/source-directory-tarballs/training/meta-textgeneration/prepack/inference-meta-textgeneration/v1.2.0/sourcedir.tar.gz",
    "TrainingArtifactUri": "s3://jumpstart-private-cache-prod-us-east-1/meta-training/v1.1.0/train-meta-textgeneration-llama-3-8b.tar.gz",
    "InferenceEnvironmentVariables": [
        {
            "Name": "SAGEMAKER_PROGRAM",
            "Type": "text",
            "Default": "inference.py",
            "Scope": "container",
            "RequiredForModelClass": True,
        },
        {
            "Name": "SAGEMAKER_SUBMIT_DIRECTORY",
            "Type": "text",
            "Default": "/opt/ml/model/code",
            "Scope": "container",
            "RequiredForModelClass": False,
        },
        {
            "Name": "SAGEMAKER_CONTAINER_LOG_LEVEL",
            "Type": "text",
            "Default": "20",
            "Scope": "container",
            "RequiredForModelClass": False,
        },
        {
            "Name": "SAGEMAKER_MODEL_SERVER_TIMEOUT",
            "Type": "text",
            "Default": "3600",
            "Scope": "container",
            "RequiredForModelClass": False,
        },
        {
            "Name": "ENDPOINT_SERVER_TIMEOUT",
            "Type": "int",
            "Default": 3600,
            "Scope": "container",
            "RequiredForModelClass": True,
        },
        {
            "Name": "MODEL_CACHE_ROOT",
            "Type": "text",
            "Default": "/opt/ml/model",
            "Scope": "container",
            "RequiredForModelClass": True,
        },
        {
            "Name": "SAGEMAKER_ENV",
            "Type": "text",
            "Default": "1",
            "Scope": "container",
            "RequiredForModelClass": True,
        },
        {
            "Name": "HF_MODEL_ID",
            "Type": "text",
            "Default": "/opt/ml/model",
            "Scope": "container",
            "RequiredForModelClass": True,
        },
        {
            "Name": "OPTION_GPU_MEMORY_UTILIZATION",
            "Type": "text",
            "Default": "0.85",
            "Scope": "container",
            "RequiredForModelClass": True,
        },
        {
            "Name": "SAGEMAKER_MODEL_SERVER_WORKERS",
            "Type": "int",
            "Default": 1,
            "Scope": "container",
            "RequiredForModelClass": True,
        },
    ],
    "DefaultInferenceInstanceType": "ml.g5.12xlarge",
    "SupportedInferenceInstanceTypes": [
        "ml.g5.12xlarge",
        "ml.g5.24xlarge",
        "ml.g5.2xlarge",
        "ml.g5.48xlarge",
        "ml.g5.4xlarge",
        "ml.g5.8xlarge",
        "ml.g6.12xlarge",
        "ml.g6.2xlarge",
        "ml.p4d.24xlarge",
        "ml.p5.48xlarge",
    ],
    "DefaultTrainingInstanceType": "ml.g5.12xlarge",
    "SupportedTrainingInstanceTypes": [
        "ml.g4dn.12xlarge",
        "ml.g5.12xlarge",
        "ml.g5.24xlarge",
        "ml.g5.48xlarge",
        "ml.p3dn.24xlarge",
    ],
    "InferenceVolumeSize": 256,
    "TrainingVolumeSize": 256,
    "InferenceEnableNetworkIsolation": True,
    "TrainingEnableNetworkIsolation": True,
    "DefaultTrainingDatasetUri": "s3://jumpstart-cache-prod-us-east-1/training-datasets/sec_amazon/",
    "ValidationSupported": True,
    "FineTuningSupported": True,
    "ResourceNameBase": "meta-textgeneration-llama-3-8b",
    "GatedBucket": True,
    "TrainingInstanceTypeVariants": {
        "Variants": {
            "g4dn": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04",
                    "GatedModelEnvVarUri": "s3://jumpstart-private-cache-prod-us-east-1/meta-training/g4dn/v1.0.0/train-meta-textgeneration-llama-3-8b.tar.gz",
                }
            },
            "g5": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04",
                    "GatedModelEnvVarUri": "s3://jumpstart-private-cache-prod-us-east-1/meta-training/g5/v1.0.0/train-meta-textgeneration-llama-3-8b.tar.gz",
                }
            },
            "g6": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "g6e": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "local_gpu": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "p2": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "p3": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "p3dn": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04",
                    "GatedModelEnvVarUri": "s3://jumpstart-private-cache-prod-us-east-1/meta-training/p3dn/v1.0.0/train-meta-textgeneration-llama-3-8b.tar.gz",
                }
            },
            "p4d": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "p4de": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "p5": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "p5e": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
            "p5en": {
                "Properties": {
                    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04"
                }
            },
        }
    },
    "HostingArtifactS3DataType": "S3Prefix",
    "HostingArtifactCompressionType": "None",
    "DynamicContainerDeploymentSupported": True,
    "InferenceConfigs": {
        "tgi": {"ComponentNames": ["tgi"]},
        "lmi": {
            "ComponentNames": ["lmi"],
            "BenchmarkMetrics": {
                "ml.g6.12xlarge": [
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.18",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "15.9",
                        "Concurrency": "32",
                    },
                ]
            },
        },
        "lmi-optimized": {
            "ComponentNames": ["lmi-optimized"],
            "BenchmarkMetrics": {
                "ml.g5.12xlarge": [
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.21",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "187.3",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.22",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "113.6",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.23",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "80.8",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.26",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "52.5",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.38",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "32.6",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.48",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "20.1",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "4.89",
                        "Concurrency": "64",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "15.0",
                        "Concurrency": "64",
                    },
                ],
                "ml.g5.2xlarge": [
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.16",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "83.2",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.20",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "69.5",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.20",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "63.2",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.21",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "50.5",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "1.60",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "63.4",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "4.89",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "145.1",
                        "Concurrency": "32",
                    },
                ],
                "ml.g6.12xlarge": [
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.14",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "134.4",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.16",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "98.3",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.17",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "72.2",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.19",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "47.7",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.26",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "29.0",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "3.78",
                        "Concurrency": "64",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "12.5",
                        "Concurrency": "64",
                    },
                ],
                "ml.g6.2xlarge": [
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.22",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "47.4",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.27",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "39.6",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.29",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "35.0",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.55",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "30.1",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "3.77",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "52.5",
                        "Concurrency": "16",
                    },
                ],
                "ml.p4d.24xlarge": [
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.06",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "186.2",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.06",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "185.9",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.06",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "183.8",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.06",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "177.6",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.07",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "162.6",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.07",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "139.7",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.08",
                        "Concurrency": "64",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "111.9",
                        "Concurrency": "64",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.38",
                        "Concurrency": "128",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "121.2",
                        "Concurrency": "128",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "2.06",
                        "Concurrency": "256",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "220.3",
                        "Concurrency": "256",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "4.71",
                        "Concurrency": "512",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "558.7",
                        "Concurrency": "512",
                    },
                ],
                "ml.p5.48xlarge": [
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.03",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "316.5",
                        "Concurrency": "1",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.03",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "333.3",
                        "Concurrency": "2",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.03",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "320.5",
                        "Concurrency": "4",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.03",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "317.5",
                        "Concurrency": "8",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.04",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "292.4",
                        "Concurrency": "16",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.04",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "261.1",
                        "Concurrency": "32",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.04",
                        "Concurrency": "64",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "210.1",
                        "Concurrency": "64",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.04",
                        "Concurrency": "128",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "161.8",
                        "Concurrency": "128",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "0.32",
                        "Concurrency": "256",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "170.9",
                        "Concurrency": "256",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "1.84",
                        "Concurrency": "512",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "268.8",
                        "Concurrency": "512",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "2.53",
                        "Concurrency": "768",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "324.7",
                        "Concurrency": "768",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "2.89",
                        "Concurrency": "1024",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "460.8",
                        "Concurrency": "1024",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "2.70",
                        "Concurrency": "1280",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "526.3",
                        "Concurrency": "1280",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "3.90",
                        "Concurrency": "1536",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "1052.6",
                        "Concurrency": "1536",
                    },
                    {
                        "Name": "latency",
                        "Unit": "sec",
                        "Value": "4.45",
                        "Concurrency": "1792",
                    },
                    {
                        "Name": "throughput",
                        "Unit": "tokens/sec",
                        "Value": "1612.9",
                        "Concurrency": "1792",
                    },
                ],
            },
            "AccelerationConfigs": [
                {"Type": "Compilation", "Enabled": False},
                {"Type": "Speculative-Decoding", "Enabled": True},
                {"Type": "Quantization", "Enabled": False},
            ],
        },
    },
    "InferenceConfigComponents": {
        "tgi": {
            "HostingEcrSpecs": {
                "Framework": "huggingface-llm",
                "FrameworkVersion": "2.0.0",
                "PyVersion": "py310",
            },
            "HostingScriptUri": s3_artifact_uri,
            "HostingUseScriptUri": False,
            "InferenceDependencies": [],
            "HostingArtifactUri": s3_artifact_uri,
            "HostingArtifactS3DataType": "S3Prefix",
            "HostingArtifactCompressionType": "None",
            "DefaultInferenceInstanceType": "ml.g5.12xlarge",
            "SupportedInferenceInstanceTypes": [
                "ml.g5.12xlarge",
                "ml.g5.24xlarge",
                "ml.g5.2xlarge",
                "ml.g5.48xlarge",
                "ml.g5.4xlarge",
                "ml.g5.8xlarge",
                "ml.g6.12xlarge",
                "ml.g6.2xlarge",
                "ml.p4d.24xlarge",
                "ml.p5.48xlarge",
            ],
            "HostingInstanceTypeVariants": {
                "Variants": {
                    "g4dn": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "g5": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "g6": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "g6e": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "local_gpu": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p2": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p3": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p3dn": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p4d": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p4de": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p5": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p5e": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "p5en": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04"
                        }
                    },
                    "ml.g5.12xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {
                                "SM_NUM_GPUS": "4",
                                "MAX_BATCH_PREFILL_TOKENS": "16384",
                            },
                            "ResourceRequirements": {
                                "MinMemoryMb": 98304,
                                "NumAccelerators": 4,
                            },
                        }
                    },
                    "ml.g5.24xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {"SM_NUM_GPUS": "4"},
                            "ResourceRequirements": {
                                "MinMemoryMb": 196608,
                                "NumAccelerators": 4,
                            },
                        }
                    },
                    "ml.g5.48xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {"SM_NUM_GPUS": "8"},
                            "ResourceRequirements": {
                                "MinMemoryMb": 393216,
                                "NumAccelerators": 8,
                            },
                        }
                    },
                    "ml.p4d.24xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {
                                "SM_NUM_GPUS": "8",
                                "MAX_BATCH_PREFILL_TOKENS": "16384",
                            },
                            "ResourceRequirements": {
                                "MinMemoryMb": 589824,
                                "NumAccelerators": 8,
                            },
                        }
                    },
                    "ml.p5.48xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {
                                "OPTION_GPU_MEMORY_UTILIZATION": "0.95"
                            }
                        }
                    },
                    "ml.g5.2xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 16384,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.4xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 32768,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.8xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 65536,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                }
            },
            "InferenceVolumeSize": 256,
            "InferenceEnableNetworkIsolation": True,
            "HostingResourceRequirements": {"MinMemoryMb": 98304, "NumAccelerators": 4},
            "InferenceEnvironmentVariables": [
                {
                    "Name": "SAGEMAKER_PROGRAM",
                    "Type": "text",
                    "Default": "inference.py",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_SUBMIT_DIRECTORY",
                    "Type": "text",
                    "Default": "/opt/ml/model/code",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "SAGEMAKER_CONTAINER_LOG_LEVEL",
                    "Type": "text",
                    "Default": "20",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "SAGEMAKER_MODEL_SERVER_TIMEOUT",
                    "Type": "text",
                    "Default": "3600",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "ENDPOINT_SERVER_TIMEOUT",
                    "Type": "int",
                    "Default": 3600,
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "MODEL_CACHE_ROOT",
                    "Type": "text",
                    "Default": "/opt/ml/model",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_ENV",
                    "Type": "text",
                    "Default": "1",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "HF_MODEL_ID",
                    "Type": "text",
                    "Default": "/opt/ml/model",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "OPTION_GPU_MEMORY_UTILIZATION",
                    "Type": "text",
                    "Default": "0.85",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SM_NUM_GPUS",
                    "Type": "text",
                    "Default": "1",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "MAX_INPUT_LENGTH",
                    "Type": "text",
                    "Default": "4095",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "MAX_TOTAL_TOKENS",
                    "Type": "text",
                    "Default": "4096",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "MAX_BATCH_PREFILL_TOKENS",
                    "Type": "text",
                    "Default": "8192",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "MAX_CONCURRENT_REQUESTS",
                    "Type": "text",
                    "Default": "512",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_MODEL_SERVER_WORKERS",
                    "Type": "int",
                    "Default": 1,
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
            ],
            "DefaultPayloads": {
                "meaningOfLife": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {
                        "generated_text": "[0].generated_text",
                        "input_logprobs": "[0].details.prefill[*].logprob",
                    },
                    "Body": {
                        "inputs": "I believe the meaning of life is",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                            "decoder_input_details": True,
                            "details": True,
                        },
                    },
                },
                "theoryOfRelativity": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "[0].generated_text"},
                    "Body": {
                        "inputs": "Simply put, the theory of relativity states that ",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
                "teamMessage": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "[0].generated_text"},
                    "Body": {
                        "inputs": "A brief message congratulating the team on the launch:\n\nHi everyone,\n\nI just ",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
                "englishToFrench": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "[0].generated_text"},
                    "Body": {
                        "inputs": "Translate English to French:\nsea otter => loutre de mer\npeppermint => menthe poivrée\nplush girafe => girafe peluche\ncheese =>",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
            },
            "ModelDataDownloadTimeout": 1200,
            "ContainerStartupHealthCheckTimeout": 1200,
            "Dependencies": [],
            "HostingEcrUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubuntu22.04",
            "SageMakerSdkPredictorSpecifications": {
                "SupportedContentTypes": ["application/json"],
                "SupportedAcceptTypes": ["application/json"],
                "DefaultContentType": "application/json",
                "DefaultAcceptType": "application/json",
            },
            "Capabilities": [],
        },
        "lmi": {
            "HostingEcrSpecs": {
                "Framework": "djl-lmi",
                "FrameworkVersion": "0.28.0",
                "PyVersion": "py310",
            },
            "HostingScriptUri": s3_artifact_uri,
            "HostingUseScriptUri": False,
            "InferenceDependencies": [],
            "HostingArtifactUri": s3_artifact_uri,
            "HostingArtifactS3DataType": "S3Prefix",
            "HostingArtifactCompressionType": "None",
            "DefaultInferenceInstanceType": "ml.g5.12xlarge",
            "SupportedInferenceInstanceTypes": [
                "ml.g5.12xlarge",
                "ml.g5.24xlarge",
                "ml.g5.2xlarge",
                "ml.g5.48xlarge",
                "ml.g5.4xlarge",
                "ml.g5.8xlarge",
                "ml.g6.12xlarge",
                "ml.g6.2xlarge",
                "ml.p4d.24xlarge",
                "ml.p5.48xlarge",
            ],
            "HostingInstanceTypeVariants": {
                "Variants": {
                    "g4dn": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "g5": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "g6": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "g6e": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "local_gpu": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p2": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p3": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p3dn": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p4d": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p4de": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p5": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p5e": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p5en": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "ml.p4d.24xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {
                                "OPTION_TENSOR_PARALLEL_DEGREE": "1"
                            },
                            "ResourceRequirements": {
                                "MinMemoryMb": 589824,
                                "NumAccelerators": 8,
                            },
                        }
                    },
                    "ml.p5.48xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {
                                "OPTION_TENSOR_PARALLEL_DEGREE": "1",
                                "OPTION_GPU_MEMORY_UTILIZATION": "0.95",
                            }
                        }
                    },
                    "ml.g5.2xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 16384,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.4xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 32768,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.8xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 65536,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.12xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 98304,
                                "NumAccelerators": 4,
                            }
                        }
                    },
                    "ml.g5.24xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 196608,
                                "NumAccelerators": 4,
                            }
                        }
                    },
                    "ml.g5.48xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 393216,
                                "NumAccelerators": 8,
                            }
                        }
                    },
                }
            },
            "InferenceVolumeSize": 256,
            "InferenceEnableNetworkIsolation": True,
            "HostingResourceRequirements": {"MinMemoryMb": 98304, "NumAccelerators": 4},
            "InferenceEnvironmentVariables": [
                {
                    "Name": "SAGEMAKER_PROGRAM",
                    "Type": "text",
                    "Default": "inference.py",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_SUBMIT_DIRECTORY",
                    "Type": "text",
                    "Default": "/opt/ml/model/code",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "SAGEMAKER_CONTAINER_LOG_LEVEL",
                    "Type": "text",
                    "Default": "20",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "SAGEMAKER_MODEL_SERVER_TIMEOUT",
                    "Type": "text",
                    "Default": "3600",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "ENDPOINT_SERVER_TIMEOUT",
                    "Type": "int",
                    "Default": 3600,
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "MODEL_CACHE_ROOT",
                    "Type": "text",
                    "Default": "/opt/ml/model",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_ENV",
                    "Type": "text",
                    "Default": "1",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "HF_MODEL_ID",
                    "Type": "text",
                    "Default": "/opt/ml/model",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "OPTION_GPU_MEMORY_UTILIZATION",
                    "Type": "text",
                    "Default": "0.85",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_MODEL_SERVER_WORKERS",
                    "Type": "int",
                    "Default": 1,
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
            ],
            "DefaultPayloads": {
                "meaningOfLife": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "I believe the meaning of life is",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                            "decoder_input_details": True,
                            "details": True,
                        },
                    },
                },
                "theoryOfRelativity": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "Simply put, the theory of relativity states that ",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
                "teamMessage": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "A brief message congratulating the team on the launch:\n\nHi everyone,\n\nI just ",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
                "englishToFrench": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "Translate English to French:\nsea otter => loutre de mer\npeppermint => menthe poivrée\nplush girafe => girafe peluche\ncheese =>",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
            },
            "ModelDataDownloadTimeout": 1200,
            "ContainerStartupHealthCheckTimeout": 1200,
            "Dependencies": [],
            "HostingEcrUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124",
            "SageMakerSdkPredictorSpecifications": {
                "SupportedContentTypes": ["application/json"],
                "SupportedAcceptTypes": ["application/json"],
                "DefaultContentType": "application/json",
                "DefaultAcceptType": "application/json",
            },
            "Capabilities": [],
        },
        "lmi-optimized": {
            "HostingEcrSpecs": {
                "Framework": "djl-lmi",
                "FrameworkVersion": "0.28.0",
                "PyVersion": "py310",
            },
            "HostingScriptUri": s3_artifact_uri,
            "HostingUseScriptUri": False,
            "InferenceDependencies": [],
            "HostingArtifactUri": s3_artifact_uri,
            "HostingArtifactS3DataType": "S3Prefix",
            "HostingArtifactCompressionType": "None",
            "HostingAdditionalDataSources": {
                "speculative_decoding": [
                    {
                        "ChannelName": "draft_model",
                        "ArtifactVersion": "v3",
                        "S3DataSource": {
                            "CompressionType": "None",
                            "S3DataType": "S3Prefix",
                            "S3Uri": "s3://sagemaker-sd-models-prod-us-east-1/sagemaker-speculative-decoding-llama3-small-v3/",
                        },
                    }
                ]
            },
            "DefaultInferenceInstanceType": "ml.g5.12xlarge",
            "SupportedInferenceInstanceTypes": [
                "ml.g5.12xlarge",
                "ml.g5.24xlarge",
                "ml.g5.2xlarge",
                "ml.g5.48xlarge",
                "ml.g5.4xlarge",
                "ml.g5.8xlarge",
                "ml.g6.12xlarge",
                "ml.g6.2xlarge",
                "ml.p4d.24xlarge",
                "ml.p5.48xlarge",
            ],
            "HostingInstanceTypeVariants": {
                "Variants": {
                    "g4dn": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "g5": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "g6": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "g6e": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "local_gpu": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p2": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p3": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p3dn": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p4d": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p4de": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p5": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p5e": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "p5en": {
                        "Properties": {
                            "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124"
                        }
                    },
                    "ml.p4d.24xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {
                                "OPTION_TENSOR_PARALLEL_DEGREE": "1"
                            },
                            "ResourceRequirements": {
                                "MinMemoryMb": 589824,
                                "NumAccelerators": 8,
                            },
                        }
                    },
                    "ml.p5.48xlarge": {
                        "Properties": {
                            "EnvironmentVariables": {
                                "OPTION_TENSOR_PARALLEL_DEGREE": "1",
                                "OPTION_GPU_MEMORY_UTILIZATION": "0.95",
                            }
                        }
                    },
                    "ml.g5.2xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 16384,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.4xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 32768,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.8xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 65536,
                                "NumAccelerators": 1,
                            }
                        }
                    },
                    "ml.g5.12xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 98304,
                                "NumAccelerators": 4,
                            }
                        }
                    },
                    "ml.g5.24xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 196608,
                                "NumAccelerators": 4,
                            }
                        }
                    },
                    "ml.g5.48xlarge": {
                        "Properties": {
                            "ResourceRequirements": {
                                "MinMemoryMb": 393216,
                                "NumAccelerators": 8,
                            }
                        }
                    },
                }
            },
            "InferenceVolumeSize": 256,
            "InferenceEnableNetworkIsolation": True,
            "HostingResourceRequirements": {"MinMemoryMb": 98304, "NumAccelerators": 4},
            "InferenceEnvironmentVariables": [
                {
                    "Name": "SAGEMAKER_PROGRAM",
                    "Type": "text",
                    "Default": "inference.py",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_SUBMIT_DIRECTORY",
                    "Type": "text",
                    "Default": "/opt/ml/model/code",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "SAGEMAKER_CONTAINER_LOG_LEVEL",
                    "Type": "text",
                    "Default": "20",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "SAGEMAKER_MODEL_SERVER_TIMEOUT",
                    "Type": "text",
                    "Default": "3600",
                    "Scope": "container",
                    "RequiredForModelClass": False,
                },
                {
                    "Name": "ENDPOINT_SERVER_TIMEOUT",
                    "Type": "int",
                    "Default": 3600,
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "MODEL_CACHE_ROOT",
                    "Type": "text",
                    "Default": "/opt/ml/model",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_ENV",
                    "Type": "text",
                    "Default": "1",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "HF_MODEL_ID",
                    "Type": "text",
                    "Default": "/opt/ml/model",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "OPTION_SPECULATIVE_DRAFT_MODEL",
                    "Type": "text",
                    "Default": "/opt/ml/additional-model-data-sources/draft_model",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "OPTION_GPU_MEMORY_UTILIZATION",
                    "Type": "text",
                    "Default": "0.85",
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
                {
                    "Name": "SAGEMAKER_MODEL_SERVER_WORKERS",
                    "Type": "int",
                    "Default": 1,
                    "Scope": "container",
                    "RequiredForModelClass": True,
                },
            ],
            "DefaultPayloads": {
                "meaningOfLife": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "I believe the meaning of life is",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
                "theoryOfRelativity": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "Simply put, the theory of relativity states that ",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
                "teamMessage": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "A brief message congratulating the team on the launch:\n\nHi everyone,\n\nI just ",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
                "englishToFrench": {
                    "ContentType": "application/json",
                    "PromptKey": "inputs",
                    "OutputKeys": {"generated_text": "generated_text"},
                    "Body": {
                        "inputs": "Translate English to French:\nsea otter => loutre de mer\npeppermint => menthe poivrée\nplush girafe => girafe peluche\ncheese =>",
                        "parameters": {
                            "max_new_tokens": 64,
                            "top_p": 0.9,
                            "temperature": 0.6,
                        },
                    },
                },
            },
            "ModelDataDownloadTimeout": 1200,
            "ContainerStartupHealthCheckTimeout": 1200,
            "Dependencies": [],
            "HostingEcrUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.28.0-lmi10.0.0-cu124",
            "SageMakerSdkPredictorSpecifications": {
                "SupportedContentTypes": ["application/json"],
                "SupportedAcceptTypes": ["application/json"],
                "DefaultContentType": "application/json",
                "DefaultAcceptType": "application/json",
            },
            "Capabilities": [],
        },
    },
    "InferenceConfigRankings": {
        "overall": {
            "Description": "default",
            "Rankings": ["lmi", "lmi-optimized", "tgi"],
        }
    },
    "EncryptInterContainerTraffic": True,
    "DisableOutputCompression": True,
    "Dependencies": [],
    "MaxRuntimeInSeconds": 360000,
    "TrainingEcrUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-training:2.0.0-transformers4.28.1-gpu-py310-cu118-ubuntu20.04",
    "TrainingMetrics": [
        {
            "Name": "huggingface-textgeneration:eval-loss",
            "Regex": "eval_epoch_loss=tensor\\(([0-9\\.]+)",
        },
        {
            "Name": "huggingface-textgeneration:eval-ppl",
            "Regex": "eval_ppl=tensor\\(([0-9\\.]+)",
        },
        {
            "Name": "huggingface-textgeneration:train-loss",
            "Regex": "train_epoch_loss=([0-9\\.]+)",
        },
    ],
    "Capabilities": [],
    "NotebookLocations": {
        "DemoNotebook": "s3://jumpstart-cache-prod-us-east-1/pmm-notebooks/pmm-notebook-model-hub-text-generation-deploy.ipynb",
        "DemoNotebooks": [
            {
                "Title": "Deploy",
                "IsDefault": True,
                "S3Uri": "s3://jumpstart-cache-prod-us-east-1/pmm-notebooks/pmm-notebook-model-hub-text-generation-deploy.ipynb",
            },
            {
                "Title": "Fine-Tune: Instruction Tuning",
                "IsDefault": False,
                "S3Uri": "s3://jumpstart-cache-prod-us-east-1/pmm-notebooks/pmm-notebook-model-hub-text-generation-instruction-tuning-llama.ipynb",
            },
        ],
    },
    "ModelTypes": ["OPEN_WEIGHTS"],
    "Task": "Text Generation",
    "Framework": "meta",
    "DataType": "text",
    "ContextualHelp": {
        "HubFormatTrainData": [
            "A train and an optional validation directories. Each directory contains a TXT. ",
            " [Learn how to setup an AWS S3 bucket.](https://docs.aws.amazon.com/AmazonS3/latest/dev/UsingBucket.html)",
        ],
        "HubDefaultTrainData": [
            "Dataset: [SEC](https://www.sec.gov/edgar/searchedgar/companysearch)",
            "SEC filing contains regulatory documents that companies and issuers of securities must submit to the Securities and Exchange Commission (SEC) on a regular basis.",
            "License: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/legalcode)",
        ],
    },
}

In [43]:
# Convert the dictionary to a JSON string
import json

hub_doc = json.dumps(hub_model_dict)

In [44]:
# importing to the hub

sm_client.import_hub_content(
    HubContentName="fine-tuned-llama3-8B-0310",
    HubContentType="Model",
    DocumentSchemaVersion="2.3.0",
    HubName=HUB_NAME,
    HubContentDocument=hub_doc,
)

{'HubArn': 'arn:aws:sagemaker:us-east-1:891376962744:hub/Custom-Model-HubZ',
 'HubContentArn': 'arn:aws:sagemaker:us-east-1:891376962744:hub-content/Custom-Model-HubZ/Model/fine-tuned-llama3-8B-0310/0.0.3',
 'ResponseMetadata': {'RequestId': 'ef944d9d-51cb-4849-a632-7fc0d19f2a52',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'ef944d9d-51cb-4849-a632-7fc0d19f2a52',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '202',
   'date': 'Mon, 10 Mar 2025 18:25:56 GMT'},
  'RetryAttempts': 0}}

In [45]:
# list the models from the private hub
hub.list_models()

{'hub_content_summaries': [{'HubContentName': 'blog-model-test0213',
   'HubContentArn': 'arn:aws:sagemaker:us-east-1:891376962744:hub-content/Custom-Model-HubZ/Model/blog-model-test0213/0.0.5',
   'HubContentVersion': '0.0.5',
   'HubContentType': 'Model',
   'DocumentSchemaVersion': '2.2.0',
   'HubContentStatus': 'Available',
   'CreationTime': datetime.datetime(2025, 2, 17, 14, 51, 33, 773000, tzinfo=tzlocal())},
  {'HubContentName': 'blog-model-test0309',
   'HubContentArn': 'arn:aws:sagemaker:us-east-1:891376962744:hub-content/Custom-Model-HubZ/Model/blog-model-test0309/0.0.1',
   'HubContentVersion': '0.0.1',
   'HubContentType': 'Model',
   'DocumentSchemaVersion': '2.3.0',
   'HubContentStatus': 'Available',
   'CreationTime': datetime.datetime(2025, 3, 10, 3, 0, 53, 502000, tzinfo=tzlocal())},
  {'HubContentName': 'fine-tuned-llama3-8B-0310',
   'HubContentArn': 'arn:aws:sagemaker:us-east-1:891376962744:hub-content/Custom-Model-HubZ/Model/fine-tuned-llama3-8B-0310/0.0.3',
   

In [46]:
# deploy the model from jumpstart

# retrieve the HUB's arn
HUB_ARN = hub.describe()["HubArn"]
print(HUB_ARN)

# get the model id
model_id = hub.list_models()["hub_content_summaries"][2]["HubContentName"]
print(model_id)

# get the model version
version = hub.list_models()["hub_content_summaries"][2]["HubContentVersion"]
print(version)

# deploy the model from the hub as an endpoint
from sagemaker.jumpstart.model import JumpStartModel

model = JumpStartModel(
    model_id=model_id, model_version=version, hub_name=HUB_ARN, region=REGION
)
predictor = model.deploy(
    accept_eula=True, initial_instance_count=1, instance_type="ml.g5.12xlarge"
)

endpoint_name = predictor.endpoint_name
endpoint_name

arn:aws:sagemaker:us-east-1:891376962744:hub/Custom-Model-HubZ
fine-tuned-llama3-8B-0310
0.0.3
----------!

'meta-textgeneration-llama-3-8b-2025-03-10-18-26-04-236'

In [47]:
# invoke the endpoint

import boto3
import json

# Create a SageMaker runtime client
runtime = boto3.client("sagemaker-runtime")

# Define your endpoint name
endpoint_name = endpoint_name

# Define the input payload
payload = {
    "inputs": "Write a poem about the spring season.",
    "parameters": {
        "max_new_tokens": 256,
        "temperature": 0.7,
        "top_p": 0.9,
        "do_sample": True,
    },
}

# Convert the payload to JSON string
payload_json = json.dumps(payload)

# Invoke the endpoint
try:
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=payload_json,
        CustomAttributes="accept_eula=true",  # Required for Llama models
    )

    # Get the response body and decode it
    response_body = json.loads(response["Body"].read().decode())
    print(response_body)

except Exception as e:
    print(f"Error invoking endpoint: {str(e)}")

{'generated_text': " Your poem can be humorous, serious, or anything in between. For example, you can write about the sounds you hear, the sights you see, or the smells you experience. You can write about the spring season in general, or you can write about a specific event or moment.\nSpring is a time of rebirth and renewal. The weather is getting warmer, the flowers are blooming, and the birds are singing. It's a time when everything seems to be coming back to life after a long winter.\nThe spring season is often associated with new beginnings. It's a time when people start fresh and try new things. It's also a time when people are more likely to take risks and be adventurous.\nIf you're looking for inspiration, here are a few ideas to get you started:\n-Write about the first signs of spring, such as the appearance of crocuses or the sound of birds chirping.\n-Write about the changes you see around you, such as the trees budding or the grass turning green.\n-Write about the feelings 

### Clean up resources

In [48]:
import boto3

# Create a SageMaker client
sm_client = boto3.client("sagemaker")

# Delete the endpoint
sm_client.delete_endpoint(EndpointName=endpoint_name)

{'ResponseMetadata': {'RequestId': '231f3329-b873-4c28-b743-5f470b4abbb7',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '231f3329-b873-4c28-b743-5f470b4abbb7',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Mon, 10 Mar 2025 18:31:41 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

# Appendix

### 1. Supported Inference Parameters

---
This model supports the following inference payload parameters:

* **max_new_tokens:** Model generates text until the output length (excluding the input context length) reaches max_new_tokens. If specified, it must be a positive integer.
* **temperature:** Controls the randomness in the output. Higher temperature results in output sequence with low-probability words and lower temperature results in output sequence with high-probability words. If `temperature` -> 0, it results in greedy decoding. If specified, it must be a positive float.
* **top_p:** In each step of text generation, sample from the smallest possible set of words with cumulative probability `top_p`. If specified, it must be a float between 0 and 1.
* **return_full_text:** If True, input text will be part of the output generated text. If specified, it must be boolean. The default value for it is False.

You may specify any subset of the parameters mentioned above while invoking an endpoint. 


### Notes
- If `max_new_tokens` is not defined, the model may generate up to the maximum total tokens allowed, which is 8K for these models. This may result in endpoint query timeout errors, so it is recommended to set `max_new_tokens` when possible. For 8B and 70B models, we recommend to set `max_new_tokens` no greater than 1500 and 500 respectively, while keeping the total number of tokens less than 8K.
- In order to support a 8k context length, this model has restricted query payloads to only utilize a batch size of 1. Payloads with larger batch sizes will receive an endpoint error prior to inference.

---

### 2. Dataset formatting instruction for training

---

####  Fine-tune the Model on a New Dataset
We currently offer two types of fine-tuning: instruction fine-tuning and domain adaption fine-tuning. You can easily switch to one of the training 
methods by specifying parameter `instruction_tuned` being 'True' or 'False'.


#### 2.1. Domain adaptation fine-tuning
The Text Generation model can also be fine-tuned on any domain specific dataset. After being fine-tuned on the domain specific dataset, the model
is expected to generate domain specific text and solve various NLP tasks in that specific domain with **few shot prompting**.

Below are the instructions for how the training data should be formatted for input to the model.

- **Input:** A train and an optional validation directory. Each directory contains a CSV/JSON/TXT file. 
  - For CSV/JSON files, the train or validation data is used from the column called 'text' or the first column if no column called 'text' is found.
  - The number of files under train and validation (if provided) should equal to one, respectively. 
- **Output:** A trained model that can be deployed for inference. 

Below is an example of a TXT file for fine-tuning the Text Generation model. The TXT file is SEC filings of Amazon from year 2021 to 2022.

```Note About Forward-Looking Statements
This report includes estimates, projections, statements relating to our
business plans, objectives, and expected operating results that are “forward-
looking statements” within the meaning of the Private Securities Litigation
Reform Act of 1995, Section 27A of the Securities Act of 1933, and Section 21E
of the Securities Exchange Act of 1934. Forward-looking statements may appear
throughout this report, including the following sections: “Business” (Part I,
Item 1 of this Form 10-K), “Risk Factors” (Part I, Item 1A of this Form 10-K),
and “Management’s Discussion and Analysis of Financial Condition and Results
of Operations” (Part II, Item 7 of this Form 10-K). These forward-looking
statements generally are identified by the words “believe,” “project,”
“expect,” “anticipate,” “estimate,” “intend,” “strategy,” “future,”
“opportunity,” “plan,” “may,” “should,” “will,” “would,” “will be,” “will
continue,” “will likely result,” and similar expressions. Forward-looking
statements are based on current expectations and assumptions that are subject
to risks and uncertainties that may cause actual results to differ materially.
We describe risks and uncertainties that could cause actual results and events
to differ materially in “Risk Factors,” “Management’s Discussion and Analysis
of Financial Condition and Results of Operations,” and “Quantitative and
Qualitative Disclosures about Market Risk” (Part II, Item 7A of this Form
10-K). Readers are cautioned not to place undue reliance on forward-looking
statements, which speak only as of the date they are made. We undertake no
obligation to update or revise publicly any forward-looking statements,
whether because of new information, future events, or otherwise.
GENERAL
Embracing Our Future ...
```


#### 2.2. Instruction fine-tuning
The Text generation model can be instruction-tuned on any text data provided that the data 
is in the expected format. The instruction-tuned model can be further deployed for inference. 
Below are the instructions for how the training data should be formatted for input to the 
model.

Below are the instructions for how the training data should be formatted for input to the model.

- **Input:** A train and an optional validation directory. Train and validation directories should contain one or multiple JSON lines (`.jsonl`) formatted files. In particular, train directory can also contain an optional `*.json` file describing the input and output formats. 
  - The best model is selected according to the validation loss, calculated at the end of each epoch.
  If a validation set is not given, an (adjustable) percentage of the training data is
  automatically split and used for validation.
  - The training data must be formatted in a JSON lines (`.jsonl`) format, where each line is a dictionary
representing a single data sample. All training data must be in a single folder, however
it can be saved in multiple jsonl files. The `.jsonl` file extension is mandatory. The training
folder can also contain a `template.json` file describing the input and output formats. If no
template file is given, the following template will be used:
  ```json
  {
    "prompt": "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n### Input:\n{context}",
    "completion": "{response}"
  }
  ```
  - In this case, the data in the JSON lines entries must include `instruction`, `context` and `response` fields. If a custom template is provided it must also use `prompt` and `completion` keys to define
  the input and output templates.
  Below is a sample custom template:

  ```json
  {
    "prompt": "question: {question} context: {context}",
    "completion": "{answer}"
  }
  ```
Here, the data in the JSON lines entries must include `question`, `context` and `answer` fields. 
- **Output:** A trained model that can be deployed for inference. 

---

#### 2.3. Example fine-tuning with Domain-Adaptation dataset format
---
We provide a subset of SEC filings data of Amazon in domain adaptation dataset format. It is downloaded from publicly available [EDGAR](https://www.sec.gov/edgar/searchedgar/companysearch). Instruction of accessing the data is shown [here](https://www.sec.gov/os/accessing-edgar-data).

License: [Creative Commons Attribution-ShareAlike License (CC BY-SA 4.0)](https://creativecommons.org/licenses/by-sa/4.0/legalcode).

Please uncomment the following code to fine-tune the model on dataset in domain adaptation format.

---

In [49]:
import boto3

model_id = "meta-textgeneration-llama-3-8b"

estimator = JumpStartEstimator(
    model_id=model_id,
    environment={"accept_eula": "true"},
    instance_type="ml.g5.24xlarge",
)
estimator.set_hyperparameters(instruction_tuned="False", epoch="5")
estimator.fit(
    {
        "training": f"s3://jumpstart-cache-prod-{boto3.Session().region_name}/training-datasets/sec_amazon"
    }
)

[03/10/25 18:31:42] ERROR    Please check the troubleshooting guide for common errors:              ]8;id=952380;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=916306;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py#1050\1050]8;;\
                             https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-t                
                             roubleshooting.html#sagemaker-python-sdk-troubleshooting-create-traini                
                             ng-job                                                                                

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 │   instance_type="ml.g5.24xlarge",                                                         │
│    9 )                                                                                           │
│   10 estimator.set_hyperparameters(instruction_tuned="False", epoch="5")                         │
│ ❱ 11 estimator.fit(                                                                              │
│   12 │   {                                                                                       │
│   13 │   │   "training": f"s3://jumpstart-cache-prod-{boto3.Session().region_name}/training-d    │
│   14 │   }                                                                                       │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/jumpstart/estimator.py:718 in fit              │
│                                                                                                  │
│    715 │   │   )                                                                                 │
│    716 │   │   remove_env_var_from_estimator_kwargs_if_accept_eula_present(self.init_kwargs, ac  │
│    717 │   │                                                                                     │
│ ❱  718 │   │   return super(JumpStartEstimator, self).fit(**estimator_fit_kwargs.to_kwargs_dict  │
│    719 │                                                                                         │
│    720 │   @classmethod                                                                          │
│    721 │   def attach(                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/telemetry/telemetry_logging.py:167 in wrapper  │
│                                                                                                  │
│   164 │   │   │   │   │   caught_ex = e                                                          │
│   165 │   │   │   │   finally:                                                                   │
│   166 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 167 │   │   │   │   │   │   raise caught_ex                                                    │
│   168 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   169 │   │   │   else:                                                                          │
│   170 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/telemetry/telemetry_logging.py:138 in wrapper  │
│                                                                                                  │
│   135 │   │   │   │   start_timer = perf_counter()                                               │
│   136 │   │   │   │   try:                                                                       │
│   137 │   │   │   │   │   # Call the original function                                           │
│ ❱ 138 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   139 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   140 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   141 │   │   │   │   │   extra += f"&x-latency={round(elap

### 3. Supported Hyper-parameters for fine-tuning
---
- epoch: The number of passes that the fine-tuning algorithm takes through the training dataset. Must be an integer greater than 1. Default: 5
- learning_rate: The rate at which the model weights are updated after working through each batch of training examples. Must be a positive float greater than 0. Default: 1e-4.
- instruction_tuned: Whether to instruction-train the model or not. Must be 'True' or 'False'. Default: 'False'
- per_device_train_batch_size: The batch size per GPU core/CPU for training. Must be a positive integer. Default: 4.
- per_device_eval_batch_size: The batch size per GPU core/CPU for evaluation. Must be a positive integer. Default: 1
- max_train_samples: For debugging purposes or quicker training, truncate the number of training examples to this value. Value -1 means using all of training samples. Must be a positive integer or -1. Default: -1. 
- max_val_samples: For debugging purposes or quicker training, truncate the number of validation examples to this value. Value -1 means using all of validation samples. Must be a positive integer or -1. Default: -1. 
- max_input_length: Maximum total input sequence length after tokenization. Sequences longer than this will be truncated. If -1, max_input_length is set to the minimum of 1024 and the maximum model length defined by the tokenizer. If set to a positive value, max_input_length is set to the minimum of the provided value and the model_max_length defined by the tokenizer. Must be a positive integer or -1. Default: -1. 
- validation_split_ratio: If validation channel is none, ratio of train-validation split from the train data. Must be between 0 and 1. Default: 0.2. 
- train_data_split_seed: If validation data is not present, this fixes the random splitting of the input training data to training and validation data used by the algorithm. Must be an integer. Default: 0.
- preprocessing_num_workers: The number of processes to use for the preprocessing. If None, main process is used for preprocessing. Default: "None"
- lora_r: Lora R. Must be a positive integer. Default: 8.
- lora_alpha: Lora Alpha. Must be a positive integer. Default: 32
- lora_dropout: Lora Dropout. must be a positive float between 0 and 1. Default: 0.05. 
- int8_quantization: If True, model is loaded with 8 bit precision for training. Default for 8B: False. Default for 70B: True.
- enable_fsdp: If True, training uses Fully Sharded Data Parallelism. Default for 8B: True. Default for 70B: False.

Note 1: int8_quantization is not supported with FSDP. Also, int8_quantization = 'False' and enable_fsdp = 'False' is not supported due to CUDA memory issues for any of the g5 family instances. Thus, we recommend setting exactly one of int8_quantization or enable_fsdp to be 'True'
Note 2: Due to the size of the model, 70B model can not be fine-tuned with enable_fsdp = 'True' for any of the supported instance types.

---

### 4. Supported Instance types for fine-tuning Llama 3

---
We have tested our scripts on the following instances types for fine-tuning Llama 3:

| Model | Model ID | All Supported Instances Types for fine-tuning |
| - | - | - |
| Llama 3 8B | meta-textgeneration-llama-3-8b | ml.g5.12xlarge, ml.g5.24xlarge, ml.g5.48xlarge, ml.p3dn.24xlarge, ml.g4dn.12xlarge |
| Llama 3 8B Instruct | meta-textgeneration-llama-3-8b-instruct | ml.g5.12xlarge, ml.g5.24xlarge, ml.g5.48xlarge, ml.p3dn.24xlarge, ml.g4dn.12xlarge  |
| Llama 3 70B | meta-textgeneration-llama-3-70b | ml.g5.48xlarge, ml.p4d.24xlarge |
| Llama 3 70B Instruct | meta-textgeneration-llama-3-70b-instruct | ml.g5.48xlarge, ml.p4d.24xlarge |

Other instance types may also work to fine-tune. Note: When using p3 instances, training will be done with 32 bit precision as bfloat16 is not supported on these instances. Thus, training job would consume double the amount of CUDA memory when training on p3 instances compared to g5 instances.

---

### 5. Few notes about the fine-tuning method

---
- Fine-tuning scripts are based on [this repo](https://github.com/facebookresearch/llama-recipes/tree/main). 
- Instruction tuning dataset is first converted into domain adaptation dataset format before fine-tuning. 
- Fine-tuning scripts utilize Fully Sharded Data Parallel (FSDP) as well as Low Rank Adaptation (LoRA) method fine-tuning the models

---

### 6. Studio Kernel Dead/Creating JumpStart Model from the training Job
---
Due to the size of the Llama 70B model, training job may take several hours and the studio kernel may die during the training phase. However, during this time, training is still running in SageMaker. If this happens, you can still deploy the endpoint using the training job name with the following code:

How to find the training job name? Go to Console -> SageMaker -> Training -> Training Jobs -> Identify the training job name and substitute in the following cell. 

---

In [ ]:
from sagemaker.jumpstart.estimator import JumpStartEstimator

training_job_name = "<<Replace this with Training Job Name>>"

attached_estimator = JumpStartEstimator.attach(training_job_name, model_id)
attached_estimator.logs()
attached_estimator.deploy()

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/introduction_to_amazon_algorithms|jumpstart-foundation-models|llama-3-finetuning.ipynb)